[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/danpele/NEURAL_BIZ/blob/master/Bishop_Ch_03/NB_bishop_ch03_distributions_finance.ipynb)

# Chapter 3: Standard Distributions — Business Applications

- Fitting Normal vs Student-t to S&P 500 returns
- VaR comparison
- GMM for market regimes


## Setup


In [ ]:
!pip install yfinance numpy scipy matplotlib scikit-learn


## 1. Fitting Distributions to S&P 500 Returns


In [ ]:
import yfinance as yf
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
from scipy.stats import gaussian_kde

sp = yf.download('^GSPC', start='2000-01-01', end='2024-12-31', progress=False)
ret = np.log(sp['Close'] / sp['Close'].shift(1)).dropna().values.flatten()

# Fit Normal and Student-t
mu_n, sig_n = stats.norm.fit(ret)
df_t, mu_t, sig_t = stats.t.fit(ret)
print(f'Normal: mu={mu_n:.5f}, sigma={sig_n:.5f}')
print(f'Student-t: df={df_t:.1f}, mu={mu_t:.5f}, sigma={sig_t:.5f}')

# KDE + parametric fits
kde = gaussian_kde(ret, bw_method='silverman')
x = np.linspace(-0.06, 0.06, 500)
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(ret, bins=100, density=True, alpha=0.3, color='gray')
ax.plot(x, stats.norm.pdf(x, mu_n, sig_n), 'b-', lw=2, label='Normal')
ax.plot(x, stats.t.pdf(x, df_t, mu_t, sig_t), 'r-', lw=2, label=f'Student-t (df={df_t:.1f})')
ax.plot(x, kde(x), 'g--', lw=2, label='KDE')
ax.legend(); ax.set_title('S&P 500 Daily Returns: Distribution Fits')
plt.show()


## 2. VaR Comparison


In [ ]:
alpha = 0.01
var_norm = -stats.norm.ppf(alpha, mu_n, sig_n)
var_t = -stats.t.ppf(alpha, df_t, mu_t, sig_t)
var_hist = -np.percentile(ret, 100*alpha)

print(f'VaR at 99%:')
print(f'  Normal:     {var_norm:.4%}')
print(f'  Student-t:  {var_t:.4%}')
print(f'  Historical: {var_hist:.4%}')
print(f'\nStudent-t / Normal ratio: {var_t/var_norm:.2f}x')


## 3. GMM for Market Regimes


In [ ]:
from sklearn.mixture import GaussianMixture

ret_2d = ret.reshape(-1, 1)
gmm = GaussianMixture(n_components=3, random_state=42).fit(ret_2d)

labels = gmm.predict(ret_2d)
for k in range(3):
    mask = labels == k
    print(f'Regime {k+1}: mu={ret[mask].mean():.4%}, sigma={ret[mask].std():.4%}, ')
    print(f'          weight={gmm.weights_[k]:.1%}, n_days={mask.sum()}')
